## What are we trying to detect?

For every PDF page, we want to answer:

Does this page contain useful extractable text?

             │
        ┌────┴────┐
        │         │
       YES        NO
        │         │
        ▼         ▼
     Normal PDF  OCR candidate

But we'll go one step further.

A page can contain some text and still be problematic. So we'll collect useful diagnostics rather than simply returning True/False.

## PDF Page Inspection
##  Detecting Scanned / Non-Text PDF Pages

The objective is to inspect every PDF page before deciding whether normal
text extraction is sufficient or OCR may be required.

### Create the PDF paths and check their existance in the provided path

In [ ]:
from pathlib import Path

text_pdf_path = Path(
    "../../data/raw/pdf/azure_event_hubs_knowledge.pdf"
)

scanned_pdf_path = Path(
    "../../data/raw/pdf/azure_event_hubs_scanned.pdf"
)

print("Text PDF:")
print(text_pdf_path.resolve())
print("Exists:", text_pdf_path.exists())

print()

print("Scanned PDF:")
print(scanned_pdf_path.resolve())
print("Exists:", scanned_pdf_path.exists())

### Load the Text PDF 

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

In [ ]:
# load the text PDF using PyMuPDFLoader
text_loader = PyMuPDFLoader(str(text_pdf_path))
text_documents = text_loader.load()

### Load the Scanned PDF

In [ ]:
# Load the scanned PDF using PyMuPDFLoader
scanned_loader = PyMuPDFLoader(str(scanned_pdf_path))
scanned_documents = scanned_loader.load()

#### Create the inspection function

In [ ]:
def analyze_pdf_pages(documents):
    """
    Analyze PDF pages and return page-level diagnostics.
    """

    results = []

    for page_number, document in enumerate(documents, start=1):

        content = document.page_content.strip()

        results.append({
            "page": page_number,
            "characters": len(content),
            "has_text": bool(content),
            "metadata": document.metadata
        })

    return results

In [ ]:
# now call the function for both text and scanned documents
text_analysis = analyze_pdf_pages(text_documents)

scanned_analysis = analyze_pdf_pages(scanned_documents)

In [ ]:
text_analysis

In [ ]:
scanned_analysis

### Make the result easier to understand

In [ ]:
import pandas as pd

In [ ]:
text_df = pd.DataFrame(text_analysis)
text_df

In [ ]:
scanned_df = pd.DataFrame(scanned_analysis)

scanned_df

## This makes the difference very obvious

By this step we confirm that we have a scanned version of PDF

## Step 7 — Introduce a text threshold

Let's create a simple learning-oriented classifier.

In [ ]:
def classify_page(content, minimum_characters=50):
    """
    Simple learning-oriented classification.

    Returns:
        TEXT_AVAILABLE
        POSSIBLE_SCANNED
    """

    character_count = len(content.strip())

    if character_count >= minimum_characters:
        return "TEXT_AVAILABLE"

    return "POSSIBLE_SCANNED"

### Now test it

In [ ]:
# text documents classification
for document in text_documents:
    result = classify_page(document.page_content)
    print(result)

In [ ]:
# scanned documents test
for document in scanned_documents:
    result = classify_page(document.page_content)
    print(result)

## Step 8 — Why do we call it POSSIBLE_SCANNED?

This naming is intentional.

We should not say:

No text → definitely scanned PDF

because there are other possibilities.

For example:

No extracted text

       │
       ├── Scanned page
       │
       ├── Image-only page
       │
       ├── Extraction failure
       │
       ├── Unsupported encoding
       │
       └── Corrupted/unusual PDF

Therefore our ingestion pipeline should say:

POSSIBLE_SCANNED

rather than:
DEFINITELY_SCANNED
This is a much better engineering mindset.


## Step 9 — Build our first PDF inspection report

Let's combine everything:

In [ ]:
def generate_pdf_inspection_report(documents, minimum_characters=50):

    report = []

    for page_number, document in enumerate(documents, start=1):

        content = document.page_content.strip()
        character_count = len(content)

        if character_count >= minimum_characters:
            status = "TEXT_AVAILABLE"
        else:
            status = "POSSIBLE_SCANNED"

        report.append({
            "page": page_number,
            "characters": character_count,
            "status": status,
            "source": document.metadata.get("source"),
        })

    return report

In [ ]:
# Scanned PDF test
report = generate_pdf_inspection_report(scanned_documents)
pd.DataFrame(report)

In [ ]:
# Text PDF test
report = generate_pdf_inspection_report(text_documents)
pd.DataFrame(report)

## Step 11 — One important improvement

Our current detector only examines:

document.page_content

But for complex PDFs, that's not enough.

For our next experiment we'll inspect the PDF itself for:

                Page
                │
                ├── Text blocks
                ├── Images
                ├── Image count
                ├── Text character count
                ├── Text density
                └── Potentially suspicious pages

That gives us a much stronger picture:

                    PDF PAGE
                       │
          ┌────────────┼────────────┐
          ▼            ▼            ▼
        Text         Images       Layout
          │            │            │
          └────────────┼────────────┘
                       ▼
                Page Analysis
                       │
                       ▼
              Ingestion Decision

That will be our next experiment before OCR.

Your learning progression is now:

1.4.2 OCR & Complex PDF Ingestion

        │
        ├── 1.4.2.1 Text PDF vs Scanned PDF       ✓
        │
        ├── 1.4.2.2 Detect Non-Text Pages         ← We are here
        │
        ├── 1.4.2.3 Deep PDF Page Analysis
        │
        ├── 1.4.2.4 OCR Fundamentals
        │
        ├── 1.4.2.5 OCR Implementation
        │
        └── 1.4.2.6 OCR → LangChain Documents

Don't install an OCR engine yet. First complete the page-analysis experiment; it will make the reason for OCR much clearer and will give you a more production-oriented mental model of document ingestion.